# 04 — Forecasting, shock risk, anomalies, and uncertainty
The newest dates are the test set. Never use a random split for this forecasting problem.

In [ ]:
%pip install -q -e .
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from agridecision.evaluation.slices import regression_slice_report
from agridecision.models.training import train_model_suite

In [ ]:
features = pd.read_csv('data/processed/supervised_features.csv', parse_dates=['arrival_date'])
result = train_model_suite(features, 'artifacts', test_fraction=0.20, horizon_days=7)
print(json.dumps(result['metrics'], indent=2, default=str))

## Baseline gate
A complex model is useful only if it improves the selected metric over a simple forecast. A baseline win is a valid result, not an error.

In [ ]:
metrics = result['metrics']
model_mae = metrics['forecast_model']['mae']
baseline_mae = metrics['seasonal_naive_baseline']['mae']
print('Model beats baseline:', model_mae < baseline_mae)
print('MAE improvement %:', round((baseline_mae - model_mae) / baseline_mae * 100, 2))

In [ ]:
predictions = result['predictions']
regression_slice_report(predictions, group_column='market', minimum_rows=10)

In [ ]:
view = predictions.groupby('arrival_date')[['actual_price', 'predicted_price', 'baseline_price']].mean()
view.plot(figsize=(14, 6), title='Chronological held-out predictions')
plt.ylabel('INR per quintal')
plt.show()

In [ ]:
if {'prediction_lower_90', 'prediction_upper_90'}.issubset(predictions.columns):
    interval = predictions.groupby('arrival_date')[['actual_price', 'predicted_price', 'prediction_lower_90', 'prediction_upper_90']].mean()
    plt.figure(figsize=(14, 6))
    plt.plot(interval.index, interval['actual_price'], label='Actual')
    plt.plot(interval.index, interval['predicted_price'], label='Predicted')
    plt.fill_between(interval.index, interval['prediction_lower_90'], interval['prediction_upper_90'], alpha=.2, label='90% interval')
    plt.legend(); plt.show()